# Transforming Our Full-Stack App into a Multimodal RAG Application

## How to Upgrade the PDF Q&A App to Understand Both Text AND Images

In our previous notebooks, we built two things:

1. **A Full-Stack PDF App** (folders `052-pdfapp-LC-backend` and `053-pdfapp-LC-frontend`) that lets users upload PDFs, store them in AWS S3, and ask questions about them using a LangChain 1.0 RAG Agent.

2. **A Multimodal RAG App** (notebook `ZZZ-MULTIMODAL-APP-WITH-LC1`) that can process PDFs containing text, tables, AND images.

**Now the big question is: How do we combine these two?** How do we take our full-stack app and upgrade it so it can understand images inside PDFs, not just text?

This notebook will explain step by step, in simple terms, exactly what changes are needed.

---
## Table of Contents

1. The Big Picture: What Needs to Change?
2. Understanding the Current App (Regular RAG)
3. Understanding the Target (Multimodal RAG)
4. The Good News: The Frontend Barely Changes!
5. Backend Change 1: New Dependencies
6. Backend Change 2: Replace the PDF Loader
7. Backend Change 3: Add Image Summarization with GPT-4o
8. Backend Change 4: Replace the Vector Store
9. Backend Change 5: Update the RAG Agent
10. Backend Change 6: Rewrite the `/ask` Endpoint
11. Complete Before & After Code Comparison
12. Summary of All Changes

---
<a id='section1'></a>
## 1. The Big Picture: What Needs to Change?

Here is the most important thing to understand:

> **The frontend stays almost the same. All the real changes happen in the backend.**

Why? Because the frontend's job is simple: send a question, receive an answer, display it. It doesn't care whether the backend used regular RAG or multimodal RAG to find that answer. The "magic" of understanding images happens entirely on the server side.

### Restaurant Analogy

Remember our restaurant analogy from the previous notebook?

- **Before (Regular RAG)**: The kitchen (backend) could only read the text of recipes. If a recipe had a photo showing how the dish should look, the kitchen ignored it.

- **After (Multimodal RAG)**: The kitchen now has a **food photographer consultant** (GPT-4o vision) who can look at photos and describe what they show. Now when a customer asks "What does this dish look like?", the kitchen can answer!

The dining room (frontend) doesn't change. The waiter still carries questions and answers back and forth the same way.

### Visual Overview of Changes

```
┌──────────────────────────────────────────────────────────────────────────┐
│                        WHAT CHANGES?                                     │
│                                                                          │
│   Frontend (Next.js)          Backend (FastAPI)                          │
│   ┌─────────────────┐         ┌──────────────────────────────────────┐   │
│   │                 │         │                                      │   │
│   │  NO CHANGES     │         │  LOTS OF CHANGES:                    │   │
│   │  NEEDED!        │         │                                      │   │
│   │                 │         │  1. New dependencies                 │   │
│   │  Same question  │         │  2. Text/table extraction (API)      │   │
│   │  input, same    │         │  3. Page rendering (pymupdf)         │   │
│   │  answer display │         │  4. Image summarization (GPT-4o)     │   │
│   │                 │         │  5. Multi-Vector Retriever           │   │
│   │                 │         │  6. Updated RAG agent                │   │
│   │                 │         │  7. Rewritten /ask endpoint          │   │
│   │                 │         │  8. config.py (new API key)          │   │
│   │                 │         │                                      │   │
│   └─────────────────┘         └──────────────────────────────────────┘   │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

---
<a id='section2'></a>
## 2. Understanding the Current App (Regular RAG)

Let's first understand how the current app processes a PDF when a user asks a question.

### Current Flow (Text-Only RAG)

When a user clicks "Ask" on a PDF, here is what happens in the backend (`routers/pdfs.py`):

```
User asks question
      │
      ▼
1. Download PDF from S3
      │
      ▼
2. Load PDF with PyPDFLoader          ◄── Only reads TEXT
      │
      ▼
3. Split text into chunks              ◄── Simple text splitting
      │
      ▼
4. Create embeddings                   ◄── Embeddings of raw text chunks
      │
      ▼
5. Store in InMemoryVectorStore        ◄── Simple vector store
      │
      ▼
6. Agent searches and answers          ◄── Only has text to work with
```

### The Current Code

Here is the current `create_pdf_rag_agent` function from `routers/pdfs.py`:

```python
def create_pdf_rag_agent(pdf_path: str):
    # Load the PDF (TEXT ONLY)
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True
    )
    all_splits = text_splitter.split_documents(documents)

    # Create embeddings and vector store
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents=all_splits)

    # Create a search tool
    @tool
    def search_pdf(query: str) -> str:
        """Search the PDF document for relevant information."""
        results = vector_store.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])

    # Create the agent
    agent = create_agent(
        model="gpt-4o-mini",
        tools=[search_pdf],
        system_prompt="""You are a helpful assistant that answers questions
        about a PDF document. Use the search_pdf tool to find relevant
        information in the document."""
    )

    return agent
```

### What's the Problem?

The problem is in **Step 2**: `PyPDFLoader` only extracts text. If your PDF has:

| Content Type | Does PyPDFLoader Extract It? |
|--------------|----------------------------|
| Plain text   | Yes |
| Tables       | Partially (loses structure) |
| Images       | **No! Completely ignored** |
| Charts       | **No! Completely ignored** |
| Diagrams     | **No! Completely ignored** |

So if a user asks "What does the chart on page 3 show?", the current app **cannot answer** because it never even saw the chart!

---
<a id='section3'></a>
## 3. Understanding the Target (Multimodal RAG)

Now let's see what the multimodal version does differently.

### Target Flow (Multimodal RAG)

```
User asks question
      |
      v
1. Download PDF from S3
      |
      v
2a. Send PDF to Unstructured API        <-- Reads TEXT + TABLES
      |
      +-- Text elements
      +-- Table elements
      |
2b. Render pages with pymupdf           <-- Captures EVERYTHING visually
      |
      +-- Page images (as base64)
      |
      v
3. Summarize everything (GPT-4o-mini)   <-- NEW STEP!
      |
      +-- Text   -> Summarized
      +-- Tables -> Summarized
      +-- Page images -> Described
      |
      v
4. Multi-Vector Retriever               <-- UPGRADED vector store
      |
      +-- Vector Store: stores SUMMARIES (for searching)
      +-- Doc Store: stores ORIGINALS (for answering)
      |
      v
5. Agent searches and answers            <-- Now has text + tables + images!
```

### Why Two Tools for PDF Extraction?

We use a **two-tool approach** because each tool is best at different things:

| Tool | Best At | Used For |
|------|---------|----------|
| **Unstructured API** | Extracting structured text and tables | Text and table extraction |
| **pymupdf** | Rendering full pages as images | Capturing charts, diagrams, photos — everything visual |

The Unstructured API is great at parsing text and tables, but it can miss images — especially vector-based charts (drawn with shapes, not embedded as image files). By rendering each page as an image with pymupdf, we guarantee that GPT-4o-mini sees **everything** on the page.

### The Key Differences

| Aspect | Regular RAG (Current) | Multimodal RAG (Target) |
|--------|----------------------|------------------------|
| **Text/Table Processing** | `PyPDFLoader` (local) | `UnstructuredClient` (hosted API) |
| **Image Processing** | None | `pymupdf` page rendering + GPT-4o-mini vision |
| **What's extracted** | Text only | Text + Tables + Full page images |
| **Summarization** | None | GPT-4o-mini for all content types |
| **Vector Store** | `InMemoryVectorStore` | `MultiVectorRetriever` with Chroma |
| **What's searchable** | Raw text chunks | Summaries of text, tables, AND page descriptions |
| **Image questions** | Cannot answer | Can answer! |
| **Dependencies** | `pypdf`, `langchain` | `unstructured-client`, `chromadb`, `pillow`, `pymupdf` |
| **System deps** | None | None (all pip-installable) |

---
<a id='section4'></a>
## 4. The Good News: The Frontend Barely Changes!

This is the best part. Let's look at why the frontend doesn't need changes.

### The Frontend's Job

The frontend (`components/pdf.js`) does only 3 things:

1. **Collects** the user's question (a text input)
2. **Sends** it to the backend (`POST /pdfs/{id}/ask`)
3. **Displays** the answer (a text paragraph)

```javascript
// This is what the frontend sends:
{
    "question": "What does the chart on page 3 show?"
}

// This is what the frontend receives:
{
    "pdf_id": 5,
    "pdf_name": "financial-report.pdf",
    "question": "What does the chart on page 3 show?",
    "answer": "The chart on page 3 shows revenue growth of 25% year over year..."
}
```

Notice: the request and response format is **exactly the same** whether we use regular RAG or multimodal RAG! The frontend doesn't know or care how the backend found the answer.

### Analogy: Ordering at a Restaurant

Imagine you're at a restaurant and you ask the waiter:

> "What ingredients are in the chef's special?"

The waiter goes to the kitchen and comes back with the answer. **You don't care whether the chef:**
- Read the recipe text
- Looked at a photo of the dish
- Analyzed a chart of nutritional information

You just care about getting a good answer. Same with our frontend!

### The Only Optional Frontend Change

If you wanted to be nice to the user, you could update the placeholder text to hint that they can ask about images:

```javascript
// BEFORE (current)
placeholder="Ask a question about this PDF..."

// AFTER (optional improvement)
placeholder="Ask about text, tables, or images in this PDF..."
```

But this is purely cosmetic. The app works perfectly without it.

---
<a id='section5'></a>
## 5. Backend Change 1: New Dependencies

The first change is adding new Python libraries that enable multimodal processing.

### What to Add to `pyproject.toml`

```toml
# BEFORE (current dependencies)
dependencies = [
    "fastapi (>=0.128.0,<0.129.0)",
    "uvicorn[standard] (>=0.40.0,<0.41.0)",
    "alembic (>=1.18.2,<2.0.0)",
    "psycopg2-binary (>=2.9.11,<3.0.0)",
    "pydantic-settings (>=2.12.0,<3.0.0)",
    "boto3 (>=1.42.37,<2.0.0)",
    "python-multipart (>=0.0.22,<0.0.23)",
    "langchain (>=1.0.0,<2.0.0)",
    "langchain-openai (>=1.0.0,<2.0.0)",
    "langchain-community (>=0.4.0,<1.0.0)",
    "langchain-text-splitters (>=1.0.0,<2.0.0)",
    "pypdf (>=5.0.0,<6.0.0)",
    "requests (>=2.32.0,<3.0.0)"
]

# AFTER (with new multimodal dependencies)
dependencies = [
    # ... keep all existing dependencies ...
    # ADD these new ones:
    "langchain-chroma (>=1.1.0,<2.0.0)",          # Vector database for Multi-Vector Retriever
    "pillow (>=12.1.0,<13.0.0)",                   # Image processing
    "unstructured-client (>=0.42.10,<0.43.0)",     # API client for Unstructured hosted service
    "langchain-unstructured (>=1.0.1,<2.0.0)",     # LangChain integration with Unstructured
    "pymupdf (>=1.25.0,<2.0.0)",                   # Renders PDF pages as images for GPT-4o
]
```

### No System Dependencies Needed!

All new dependencies are pure pip-installable packages:

- The **Unstructured API** sends your PDF to Unstructured's servers for text/table extraction — no local `poppler` or `tesseract` needed.
- **pymupdf** bundles its own PDF rendering engine — no system-level PDF libraries required.

This means:
- No `brew install poppler tesseract` on macOS
- No `sudo apt install poppler-utils tesseract-ocr` on Linux
- Simpler setup, works the same on every machine

### Environment Variable

You will need an **Unstructured API key** in your `.env` file:

```
UNSTRUCTURED_API_KEY=your_api_key_here
```

You can get a free API key at [unstructured.io](https://unstructured.io).

### Why Each New Dependency?

| Package | Why We Need It |
|---------|----------------|
| `unstructured-client` | API client that sends PDFs to the Unstructured hosted service for extraction of text and tables |
| `langchain-unstructured` | LangChain integration layer for the Unstructured API |
| `langchain-chroma` | Provides the Chroma vector database needed by `MultiVectorRetriever` |
| `pillow` | Handles image processing |
| `pymupdf` | Renders PDF pages as images (PNG) so GPT-4o can see everything on each page — charts, diagrams, photos, etc. |

---
<a id='section6'></a>
## 6. Backend Change 2: Replace the PDF Loader

This is the most fundamental change. We replace the simple `PyPDFLoader` with a **two-tool approach**: the Unstructured API for text/tables, and pymupdf for page images.

### BEFORE: PyPDFLoader (Text Only)

```python
from langchain_community.document_loaders import PyPDFLoader

# Only extracts text - ignores images!
loader = PyPDFLoader(pdf_path)
documents = loader.load()
```

### AFTER: Unstructured API + pymupdf (Text + Tables + Page Images)

```python
import base64
import fitz  # pymupdf
from unstructured_client import UnstructuredClient
from config import get_settings

def extract_pdf_elements(pdf_path: str):
    """
    Extract text, tables, and images from a PDF file.

    - Text and tables: extracted via the Unstructured API
    - Page images: rendered via pymupdf (guarantees GPT-4o sees everything)

    Why two tools? The Unstructured API is great at structured text and tables,
    but can miss images — especially vector-based charts drawn with shapes.
    pymupdf renders each page as an image, so nothing is missed.
    """
    settings = get_settings()
    client = UnstructuredClient(api_key_auth=settings.UNSTRUCTURED_API_KEY)

    with open(pdf_path, "rb") as f:
        file_content = f.read()

    # Use Unstructured API for text and table extraction
    response = client.general.partition(
        request={
            "partition_parameters": {
                "files": {
                    "content": file_content,
                    "file_name": os.path.basename(pdf_path),
                },
                "strategy": "hi_res",
                "infer_table_structure": True,
            }
        }
    )

    # Without chunking, the API returns granular element types
    text_types = {
        "NarrativeText", "Title", "UncategorizedText",
        "ListItem", "Header", "Footer", "FigureCaption",
    }

    raw_texts = []
    table_elements = []

    for element in response.elements:
        el_type = element.get("type", "")
        text = element.get("text", "")

        if el_type in text_types and text.strip():
            raw_texts.append(text)
        elif el_type == "Table":
            table_elements.append(text)

    # Group small text elements into ~2000-char chunks
    text_elements = []
    current_chunk = ""
    for text in raw_texts:
        if len(current_chunk) + len(text) > 2000 and current_chunk:
            text_elements.append(current_chunk.strip())
            current_chunk = text
        else:
            current_chunk += "\n\n" + text if current_chunk else text
    if current_chunk.strip():
        text_elements.append(current_chunk.strip())

    # Render each PDF page as an image using pymupdf
    image_base64_list = []
    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        img_b64 = base64.b64encode(img_bytes).decode("utf-8")
        image_base64_list.append(img_b64)
    doc.close()

    return text_elements, table_elements, image_base64_list
```

### What Changed and Why?

| Before | After | Why |
|--------|-------|-----|
| `PyPDFLoader` (local, text-only) | `UnstructuredClient` (API) | Can extract tables with structure preserved |
| No image extraction at all | `pymupdf` renders pages as images | GPT-4o sees everything: charts, photos, diagrams |
| Returns a list of documents | Returns three separate lists | Each content type needs different processing |
| Basic text chunking | Local grouping into ~2000-char chunks | No API-side chunking needed |

### Why NOT Use the Unstructured API for Image Extraction?

You might wonder: the Unstructured API has an `extract_image_block_types` parameter — why not use it?

In practice, this parameter has **two problems**:

1. **Vector-based charts are missed**: Many PDF charts are drawn with vector shapes (rectangles, lines), not embedded as raster images. The API doesn't detect these as "Image" elements.

2. **Chunking conflicts**: If you enable `chunking_strategy` (to group text), Image elements get absorbed into `CompositeElement` chunks and their base64 data is lost. But if you disable chunking, you get many tiny text fragments.

The **pymupdf approach** solves both problems: it renders each page as a pixel image, so GPT-4o sees everything — vector charts, raster photos, diagrams, text overlays, everything.

### How It Works

```
Your Backend
     │
     ├──► Unstructured API
     │         │
     │         ├── Extracts text elements (NarrativeText, Title, etc.)
     │         └── Extracts tables (with structure)
     │
     └──► pymupdf (local)
               │
               └── Renders each page at 200 DPI → PNG → base64
                   (captures charts, photos, diagrams, everything)
```

### Analogy: The Kitchen Inspector

- **Before (PyPDFLoader)**: A kitchen inspector who can only read the recipe text. If there's a photo of the dish, they skip it.

- **After (Unstructured API + pymupdf)**: You send the recipe to a text analysis lab (Unstructured API) for the written content, AND you take a high-resolution photo of each page (pymupdf) so a visual expert (GPT-4o) can examine everything that appears on the page.

---
<a id='section7'></a>
## 7. Backend Change 3: Add Image Summarization with GPT-4o-mini

This is the **most exciting** change! We now have page images rendered by pymupdf (as base64 PNG strings), and we need the AI to "look at" and describe them.

### Why Can't We Just Store Images Directly?

Great question! Here's the problem:

- Our vector store works with **text embeddings** (numbers that represent the meaning of text)
- Images are **pixels**, not text
- We can't create text embeddings from pixels

**The solution**: Ask GPT-4o-mini (which can see images!) to **describe** each page image in text. Then we store that text description.

### The New Image Summarization Code

pymupdf renders each page as a PNG image and base64-encodes it. We pass this directly to GPT-4o-mini:

```python
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage


def summarize_image(image_base64: str, model: ChatOpenAI) -> str:
    """
    Use a vision-capable model to describe a page image.

    The image_base64 parameter comes from pymupdf's page rendering
    (page -> pixmap -> PNG -> base64).
    """
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": """Describe this image in detail. Include:
                - What type of content it shows (chart, diagram, photo, etc.)
                - Any text visible in the image
                - Key data points if it's a chart or graph
                - The overall meaning or purpose of the image"""
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{image_base64}"
                }
            }
        ]
    )

    response = model.invoke([message])
    return response.content
```

### How This Works Step by Step

```
pymupdf renders each PDF page as a PNG image
        |
        v
1. Each page is rendered at 200 DPI -> PNG bytes -> base64 string
        |
        v
2. Send to GPT-4o-mini with the prompt: "Describe this image..."
        |
        v
3. GPT-4o-mini "sees" the full page and returns a text description:
   "This page shows a financial statement with a bar chart
    showing sales from 2020 to 2024. The chart shows:
    2020: ~$1M, 2021: ~$3M, 2022: ~$8M, 2023: ~$15M, 2024: ~$22M.
    There is also a GPU photo and a 33% ROI badge..."
        |
        v
4. This TEXT description gets stored in the vector store
```

This approach is more reliable than depending on the Unstructured API to extract individual images, because:
- pymupdf captures **everything** on the page (vector charts, photos, diagrams, text)
- No elements are lost due to chunking or element type classification issues
- GPT-4o-mini sees the page exactly as a human would see it

### We Also Summarize Text and Tables

In the multimodal version, we summarize ALL content types. This is because we use the **Multi-Vector Retriever** pattern where summaries are used for searching, and originals are used for answering:

```python
def summarize_text(text: str, model: ChatOpenAI) -> str:
    """Summarize a text element."""
    prompt = f"""Summarize the following text concisely 
    while preserving key information:\n\n{text}\n\nSummary:"""
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content


def summarize_table(table: str, model: ChatOpenAI) -> str:
    """Summarize a table element."""
    prompt = f"""Summarize the following table, highlighting 
    key data points and relationships:\n\n{table}\n\nSummary:"""
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content
```

### One Model for Everything: GPT-4o-mini

We use **`gpt-4o-mini` for all content types** — text, tables, and images:

| Content Type | Model | Why |
|--------------|-------|-----|
| Text | `gpt-4o-mini` | Fast, cheap, and more capable than the legacy `gpt-3.5-turbo` |
| Tables | `gpt-4o-mini` | Better reliability for structured data than `gpt-3.5-turbo` |
| Page images | `gpt-4o-mini` | Full vision support with great cost/quality tradeoff |

### Why Not GPT-3.5-turbo?

You may see older tutorials use `gpt-3.5-turbo` for text summarization. **OpenAI now treats it as legacy** and recommends `gpt-4o-mini` instead — it's more capable, multimodal (can handle text AND images), and offers similar speed at better value.

Using one model for everything also **simplifies the code** — no need for separate `text_model` and `vision_model` variables.

### Want Higher Quality for Images?

If you need more detailed image descriptions for complex documents, you can upgrade just the image step to `gpt-4o`:

```python
# Default: one model for everything
model = ChatOpenAI(model="gpt-4o-mini", max_tokens=1024)

# Optional: use gpt-4o for images if you need higher quality
# vision_model = ChatOpenAI(model="gpt-4o", max_tokens=1024)
```

### Important Note About Vision Models

You may see older tutorials mention `gpt-4-vision-preview`. **That model is deprecated!** Vision capabilities are now built into many models, including `gpt-4o-mini` and `gpt-4o`.

---
<a id='section8'></a>
## 8. Backend Change 4: Replace the Vector Store

The current app uses a simple `InMemoryVectorStore`. The multimodal version needs a more sophisticated `MultiVectorRetriever`.

### Why Do We Need a Different Vector Store?

In regular RAG:
- We store **raw text chunks** and search them directly
- The same text we search is the same text we use to answer

In multimodal RAG, we have a **two-store pattern**:
- **Store 1 (Vector Store)**: Contains **summaries** (for searching)
- **Store 2 (Doc Store)**: Contains **original content** (for answering)

```
┌─────────────────────────────┐    ┌─────────────────────────────┐
│    VECTOR STORE (Chroma)    │    │    DOC STORE (InMemory)     │
│                             │    │                             │
│  Stores: SUMMARIES          │    │  Stores: ORIGINALS          │
│  (as embeddings)            │    │  (full content)             │
│                             │    │                             │
│  "Revenue grew 25%..."      │◄──►│  Full table with all data   │
│  UUID: abc-123              │ ID │  UUID: abc-123              │
│                             │    │                             │
│  "Bar chart showing..."     │◄──►│  Full image description     │
│  UUID: def-456              │ ID │  UUID: def-456              │
└─────────────────────────────┘    └─────────────────────────────┘
```

### Why Two Stores?

- **Summaries are better for searching**: A concise summary captures the key meaning
- **Originals are better for answering**: The full content has all the details

It's like a library card catalog:
- The **catalog card** (summary) helps you find the right book
- But you read the **actual book** (original) to get the detailed answer

### BEFORE: Simple Vector Store

```python
from langchain_core.vectorstores import InMemoryVectorStore

# Simple: store raw chunks, search raw chunks
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=all_splits)
```

### AFTER: Multi-Vector Retriever

```python
import uuid
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.schema.document import Document
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma


def create_multimodal_retriever(
    text_summaries, text_elements,
    table_summaries, table_elements,
    image_summaries
):
    # Create the embedding model
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Create the two stores
    vectorstore = Chroma(
        collection_name="multimodal_summaries",
        embedding_function=embeddings
    )
    docstore = InMemoryStore()
    id_key = "doc_id"
    
    # Create the multi-vector retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key
    )
    
    # Helper function to add documents
    def add_documents(summaries, original_contents, content_type):
        if not summaries:
            return
        doc_ids = [str(uuid.uuid4()) for _ in summaries]
        summary_docs = [
            Document(
                page_content=s,
                metadata={id_key: doc_ids[i], "type": content_type}
            )
            for i, s in enumerate(summaries)
        ]
        retriever.vectorstore.add_documents(summary_docs)
        retriever.docstore.mset(list(zip(doc_ids, original_contents)))
    
    # Add all content types
    add_documents(text_summaries, text_elements, "text")
    add_documents(table_summaries, table_elements, "table")
    # For images, the summary IS the content (description)
    add_documents(image_summaries, image_summaries, "image")
    
    return retriever
```

### Understanding UUIDs

Each piece of content gets a **UUID** (a unique identifier) that links its summary to its original:

```
Summary: "Chart showing revenue growth of 25%"   ──── UUID: abc-123
                                                           │
Original: "This bar chart displays quarterly     ──── UUID: abc-123
           revenue for FY2025. Q1: $2.3M,
           Q2: $2.8M, Q3: $3.1M, Q4: $3.5M..."
```

When a search finds the summary, the UUID is used to fetch the full original content.

### Important Note: `langchain_classic`

In LangChain 1.x, several modules (`retrievers`, `storage`, `schema`) were moved from `langchain` to `langchain_classic`. That's why we import from `langchain_classic` instead of `langchain` for these three:

| Import | Package |
|--------|---------|
| `MultiVectorRetriever` | `langchain_classic.retrievers.multi_vector` |
| `Document` | `langchain_classic.schema.document` |
| `InMemoryStore` | `langchain_classic.storage` |

---
<a id='section9'></a>
## 9. Backend Change 5: Update the RAG Agent

The agent needs a small update to its search tool and system prompt so it knows about multimodal content.

### BEFORE: Text-Only Agent

```python
# Current agent - only knows about text
@tool
def search_pdf(query: str) -> str:
    """Search the PDF document for relevant information."""
    results = vector_store.similarity_search(query, k=3)
    return "\n\n".join([doc.page_content for doc in results])

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_pdf],
    system_prompt="""You are a helpful assistant that answers questions
    about a PDF document. Use the search_pdf tool to find relevant
    information in the document."""
)
```

### AFTER: Multimodal Agent

```python
# Updated agent - knows about text, tables, AND images
@tool
def search_documents(query: str) -> str:
    """
    Search the document database for relevant information.
    The database contains text, tables, and image descriptions
    from PDF documents.
    """
    results = retriever.invoke(query)     # Use retriever.invoke() instead!
    
    if not results:
        return "No relevant information found."
    
    formatted_results = []
    for i, result in enumerate(results, 1):
        if hasattr(result, 'page_content'):
            formatted_results.append(f"[Result {i}]\n{result.page_content}")
        else:
            formatted_results.append(f"[Result {i}]\n{result}")
    
    return "\n\n".join(formatted_results)

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_documents],
    system_prompt="""You are a helpful assistant that answers questions
    about documents.

    You have access to a document database that contains:
    - Text content from PDF documents
    - Table data (financial figures, statistics, etc.)
    - Descriptions of images (charts, diagrams, photos)

    When answering questions:
    1. Use the search_documents tool to find relevant information
    2. Base your answers ONLY on the retrieved information
    3. If the information isn't in the documents, say so clearly
    4. For numerical questions, quote the exact figures
    5. For image-related questions, refer to the image descriptions

    Be concise but accurate."""
)
```

### What Changed?

| Aspect | Before | After |
|--------|--------|-------|
| Tool name | `search_pdf` | `search_documents` |
| Search method | `vector_store.similarity_search()` | `retriever.invoke()` |
| Tool description | "Search the PDF document" | Mentions text, tables, AND image descriptions |
| System prompt | Basic | Detailed instructions for multimodal content |
| Result handling | Simple join | Checks for different result types |

### Why `retriever.invoke()` Instead of `similarity_search()`?

- `similarity_search()` returns matching summaries
- `retriever.invoke()` searches summaries but **returns the originals** (via UUIDs)
- This gives the agent richer, more detailed content to base its answers on

---
<a id='section10'></a>
## 10. Backend Change 6: Rewrite the `/ask` Endpoint

Finally, we need to update the `/ask` endpoint in `routers/pdfs.py` to use all the new multimodal functions.

### BEFORE: Current `/ask` Endpoint

```python
@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest, db: Session = Depends(get_db)):
    # Get the PDF from database
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        # Download PDF from S3
        temp_path = download_pdf_from_url(pdf.file)

        # Create RAG agent (text-only)
        agent = create_pdf_rag_agent(temp_path)

        # Ask the question
        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content

        return {
            "pdf_id": id,
            "pdf_name": pdf.name,
            "question": request.question,
            "answer": answer
        }
    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

### AFTER: Multimodal `/ask` Endpoint

```python
@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest, db: Session = Depends(get_db)):
    # Get the PDF from database
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        # Step 1: Download PDF from S3
        temp_path = download_pdf_from_url(pdf.file)

        # Step 2: Extract text, tables, AND images via Unstructured API
        text_elements, table_elements, image_base64_list = extract_pdf_elements(
            temp_path
        )

        # Step 3: Create summaries for all content types
        text_summaries, table_summaries, image_summaries = create_all_summaries(
            text_elements, table_elements, image_base64_list
        )

        # Step 4: Build the Multi-Vector Retriever
        retriever = create_multimodal_retriever(
            text_summaries, text_elements,
            table_summaries, table_elements,
            image_summaries
        )

        # Step 5: Create the multimodal agent
        agent = create_multimodal_rag_agent(retriever)

        # Step 6: Ask the question
        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content

        return {
            "pdf_id": id,
            "pdf_name": pdf.name,
            "question": request.question,
            "answer": answer
        }
    finally:
        # Clean up temporary PDF file
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

### Key Differences

| Step | Before | After |
|------|--------|-------|
| PDF Processing | `PyPDFLoader` (text only) | `extract_pdf_elements` via Unstructured API (text + tables + images) |
| Summarization | None | `create_all_summaries` (GPT-3.5 + GPT-4o) |
| Vector Store | `InMemoryVectorStore` | `create_multimodal_retriever` |
| Agent | Text-only agent | Multimodal-aware agent |
| Cleanup | Delete temp PDF | Delete temp PDF only (no image temp dir needed!) |

Notice how much **simpler the cleanup** is with the API approach. Since images come back as base64 strings in the API response (not saved to disk), we don't need to create or clean up a temporary image directory.

### What About Performance?

The multimodal version is **slower** because it does more work:

| Step | Regular RAG | Multimodal RAG | Extra Time |
|------|-------------|----------------|------------|
| PDF Extraction | ~1 sec (local) | ~5-15 sec (API call) | Network round-trip + server processing |
| Summarization | None | ~10-30 sec | Calling GPT for each element |
| Vector Store | ~1 sec | ~2-3 sec | More documents to embed |
| Agent Query | ~3-5 sec | ~3-5 sec | Same |
| **Total** | **~5-7 sec** | **~20-55 sec** | Worth it for multimodal! |

**Tip for production**: You would process the PDF once (when uploaded) and cache the retriever, rather than re-processing it on every question. We keep it simple here for educational purposes.

---
<a id='section11'></a>
## 11. Complete Before & After Code Comparison

Let's put it all together. Here is the complete `routers/pdfs.py` file comparison.

### BEFORE: Complete `routers/pdfs.py` (Regular RAG)

```python
from typing import List
from sqlalchemy.orm import Session
from fastapi import APIRouter, Depends, HTTPException, status, UploadFile, File
import schemas
import crud
from database import SessionLocal
from uuid import uuid4

# LangChain imports
import tempfile
import os
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

router = APIRouter(prefix="/pdfs")

# ... (CRUD endpoints stay exactly the same) ...

def download_pdf_from_url(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    temp_file.write(response.content)
    temp_file.close()
    return temp_file.name


def create_pdf_rag_agent(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=200, add_start_index=True
    )
    all_splits = text_splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents=all_splits)

    @tool
    def search_pdf(query: str) -> str:
        """Search the PDF document for relevant information."""
        results = vector_store.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])

    agent = create_agent(
        model="gpt-4o-mini",
        tools=[search_pdf],
        system_prompt="""You are a helpful assistant that answers questions
        about a PDF document. Use the search_pdf tool to find relevant
        information. Always base your answers on the document."""
    )
    return agent


@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest, 
                  db: Session = Depends(get_db)):
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        temp_path = download_pdf_from_url(pdf.file)
        agent = create_pdf_rag_agent(temp_path)
        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content
        return {
            "pdf_id": id, "pdf_name": pdf.name,
            "question": request.question, "answer": answer
        }
    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

### AFTER: Complete `routers/pdfs.py` (Multimodal RAG)

```python
from typing import List
from sqlalchemy.orm import Session
from fastapi import APIRouter, Depends, HTTPException, status, UploadFile, File
import schemas
import crud
from database import SessionLocal
from uuid import uuid4

# LangChain imports (UPDATED for multimodal)
import tempfile
import os
import uuid
import base64
import requests
import fitz  # pymupdf: renders PDF pages as images
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.schema.document import Document
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from unstructured_client import UnstructuredClient
from config import get_settings

router = APIRouter(prefix="/pdfs")

# ... (CRUD endpoints stay exactly the same) ...


def download_pdf_from_url(url: str) -> str:
    """Download a PDF from S3 URL. (Same as before - no changes needed.)"""
    response = requests.get(url)
    response.raise_for_status()
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    temp_file.write(response.content)
    temp_file.close()
    return temp_file.name


# ========== NEW: PDF EXTRACTION via Unstructured API + pymupdf ==========

def extract_pdf_elements(pdf_path: str):
    """
    Extract text, tables, and images from a PDF file.

    - Text and tables: extracted via the Unstructured API (good at structured content)
    - Page images: rendered via pymupdf and sent to GPT-4o-mini for description

    pymupdf renders each page as an image, which guarantees GPT-4o-mini sees everything
    on the page — including vector-based charts that the Unstructured API may miss.
    """
    settings = get_settings()
    client = UnstructuredClient(api_key_auth=settings.UNSTRUCTURED_API_KEY)

    with open(pdf_path, "rb") as f:
        file_content = f.read()

    # Use Unstructured API for text and table extraction
    response = client.general.partition(
        request={
            "partition_parameters": {
                "files": {
                    "content": file_content,
                    "file_name": os.path.basename(pdf_path),
                },
                "strategy": "hi_res",
                "infer_table_structure": True,
            }
        }
    )

    text_types = {
        "NarrativeText", "Title", "UncategorizedText",
        "ListItem", "Header", "Footer", "FigureCaption",
    }

    raw_texts = []
    table_elements = []

    for element in response.elements:
        el_type = element.get("type", "")
        text = element.get("text", "")

        if el_type in text_types and text.strip():
            raw_texts.append(text)
        elif el_type == "Table":
            table_elements.append(text)

    # Group small text elements into ~2000-char chunks
    text_elements = []
    current_chunk = ""
    for text in raw_texts:
        if len(current_chunk) + len(text) > 2000 and current_chunk:
            text_elements.append(current_chunk.strip())
            current_chunk = text
        else:
            current_chunk += "\n\n" + text if current_chunk else text
    if current_chunk.strip():
        text_elements.append(current_chunk.strip())

    # Render each PDF page as an image using pymupdf
    # This captures everything visible on the page: charts, diagrams, photos, etc.
    image_base64_list = []
    doc = fitz.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        img_b64 = base64.b64encode(img_bytes).decode("utf-8")
        image_base64_list.append(img_b64)
    doc.close()

    return text_elements, table_elements, image_base64_list


# ========== NEW: SUMMARIZATION FUNCTIONS ==========

def summarize_image(image_base64: str, model: ChatOpenAI) -> str:
    """Use a vision-capable model to describe a page image (base64 PNG from pymupdf)."""
    message = HumanMessage(content=[
        {"type": "text", "text": """Describe this image in detail. Include:
                - What type of content it shows (chart, diagram, photo, etc.)
                - Any text visible in the image
                - Key data points if it's a chart or graph
                - The overall meaning or purpose of the image"""},
        {"type": "image_url", "image_url": {
            "url": f"data:image/jpeg;base64,{image_base64}"
        }}
    ])
    return model.invoke([message]).content


def summarize_text(text: str, model: ChatOpenAI) -> str:
    """Summarize a text element."""
    prompt = f"Summarize concisely:\n\n{text}\n\nSummary:"
    return model.invoke([HumanMessage(content=prompt)]).content


def summarize_table(table: str, model: ChatOpenAI) -> str:
    """Summarize a table element."""
    prompt = f"Summarize this table with key data points:\n\n{table}\n\nSummary:"
    return model.invoke([HumanMessage(content=prompt)]).content


def create_all_summaries(text_elements, table_elements, image_base64_list):
    """Create summaries for all content types using gpt-4o-mini."""
    model = ChatOpenAI(model="gpt-4o-mini", max_tokens=1024)

    text_summaries = [summarize_text(t, model) for t in text_elements]
    table_summaries = [summarize_table(t, model) for t in table_elements]
    image_summaries = [summarize_image(b64, model) for b64 in image_base64_list]

    return text_summaries, table_summaries, image_summaries


# ========== NEW: MULTI-VECTOR RETRIEVER ==========

def create_multimodal_retriever(
    text_summaries, text_elements,
    table_summaries, table_elements,
    image_summaries
):
    """Create a Multi-Vector Retriever for multimodal content."""
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = Chroma(
        collection_name="multimodal_summaries",
        embedding_function=embeddings
    )
    docstore = InMemoryStore()
    id_key = "doc_id"

    retriever = MultiVectorRetriever(
        vectorstore=vectorstore, docstore=docstore, id_key=id_key
    )

    def add_docs(summaries, originals, content_type):
        if not summaries:
            return
        doc_ids = [str(uuid.uuid4()) for _ in summaries]
        summary_docs = [
            Document(page_content=s, metadata={id_key: doc_ids[i], "type": content_type})
            for i, s in enumerate(summaries)
        ]
        retriever.vectorstore.add_documents(summary_docs)
        retriever.docstore.mset(list(zip(doc_ids, originals)))

    add_docs(text_summaries, text_elements, "text")
    add_docs(table_summaries, table_elements, "table")
    add_docs(image_summaries, image_summaries, "image")

    return retriever


# ========== NEW: MULTIMODAL AGENT ==========

def create_multimodal_rag_agent(retriever):
    """Create a RAG agent that knows about text, tables, and images."""

    @tool
    def search_documents(query: str) -> str:
        """Search documents for text, table data, and image descriptions."""
        results = retriever.invoke(query)
        if not results:
            return "No relevant information found."
        formatted = []
        for i, r in enumerate(results, 1):
            content = r.page_content if hasattr(r, 'page_content') else str(r)
            formatted.append(f"[Result {i}]\n{content}")
        return "\n\n".join(formatted)

    agent = create_agent(
        model="gpt-4o-mini",
        tools=[search_documents],
        system_prompt="""You are a helpful assistant that answers questions 
        about documents. You have access to text, tables, and image 
        descriptions. Use search_documents to find relevant information. 
        Base your answers ONLY on retrieved information."""
    )
    return agent


# ========== UPDATED /ask ENDPOINT ==========

@router.post("/{id}/ask")
async def ask_pdf(id: int, request: schemas.QuestionRequest,
                  db: Session = Depends(get_db)):
    pdf = crud.read_pdf(db, id)
    if pdf is None:
        raise HTTPException(status_code=404, detail="PDF not found")

    temp_path = None
    try:
        temp_path = download_pdf_from_url(pdf.file)

        text_el, table_el, image_b64 = extract_pdf_elements(temp_path)
        text_sum, table_sum, image_sum = create_all_summaries(
            text_el, table_el, image_b64
        )
        retriever = create_multimodal_retriever(
            text_sum, text_el, table_sum, table_el, image_sum
        )
        agent = create_multimodal_rag_agent(retriever)

        response = agent.invoke(
            {"messages": [HumanMessage(content=request.question)]}
        )
        answer = response["messages"][-1].content

        return {
            "pdf_id": id, "pdf_name": pdf.name,
            "question": request.question, "answer": answer
        }
    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)
```

---
### What About the Imports?

Here is a clear comparison of what imports changed:

```python
# ========== REMOVED IMPORTS ==========
# These are no longer needed:
from langchain_community.document_loaders import PyPDFLoader      # Replaced by Unstructured API
from langchain_text_splitters import RecursiveCharacterTextSplitter # Replaced by local chunking
from langchain_core.vectorstores import InMemoryVectorStore        # Replaced by MultiVectorRetriever

# ========== ADDED IMPORTS ==========
# These are new:
import uuid                                                                    # For unique IDs
import base64                                                                  # For encoding page images
import fitz                                                                    # pymupdf: renders PDF pages as images
import requests                                                                # For downloading PDFs from S3
from langchain_openai import ChatOpenAI                                        # For summarization models
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever     # New retriever
from langchain_classic.schema.document import Document                         # For creating documents
from langchain_classic.storage import InMemoryStore                            # Doc store for originals
from langchain_chroma import Chroma                                            # Vector database
from unstructured_client import UnstructuredClient                             # API client for text/table extraction
from config import get_settings                                                # To access UNSTRUCTURED_API_KEY
```

### Why `langchain_classic`?

In LangChain 1.x, several modules were moved from the `langchain` package to `langchain_classic`. Three of our imports are affected:

| Class | Old path (broken in 1.x) | Correct path |
|-------|--------------------------|--------------|
| `MultiVectorRetriever` | `langchain.retrievers.multi_vector` | `langchain_classic.retrievers.multi_vector` |
| `Document` | `langchain.schema.document` | `langchain_classic.schema.document` |
| `InMemoryStore` | `langchain.storage` | `langchain_classic.storage` |

The other imports (`tool`, `create_agent`, `HumanMessage`) still work fine from `langchain`.

### What's NOT Needed Anymore

Several imports from the old approaches are no longer needed:

| No longer needed | Why |
|-----------------|-----|
| `import shutil` | Was used to clean up temp image directories. Page images are generated in-memory by pymupdf. |
| `from pathlib import Path` | Was used to create the image output directory. No temp directories needed. |
| `from unstructured.partition.pdf import partition_pdf` | Replaced by `UnstructuredClient` API calls for text/tables, and pymupdf for images. |
| `PyPDFLoader` | Replaced by `UnstructuredClient` for text/tables. |
| `RecursiveCharacterTextSplitter` | Replaced by local ~2000-char chunk grouping. |

---
<a id='section12'></a>
## 12. Summary of All Changes

### Complete Checklist

Here is every change needed to transform the app:

| # | File | Change | Difficulty |
|---|------|--------|------------|
| 1 | `pyproject.toml` | Add `unstructured-client`, `langchain-unstructured`, `langchain-chroma`, `pillow`, `pymupdf` | Easy |
| 2 | `.env` | Add `UNSTRUCTURED_API_KEY` | Easy |
| 3 | `config.py` | Add `UNSTRUCTURED_API_KEY: str` to Settings class | Easy |
| 4 | `routers/pdfs.py` | Update imports (add `base64`, `fitz`, `UnstructuredClient`, etc.) | Easy |
| 5 | `routers/pdfs.py` | Add `extract_pdf_elements()` function (Unstructured API for text/tables + pymupdf for page images) | Medium |
| 6 | `routers/pdfs.py` | Add summarization functions (`summarize_image`, `summarize_text`, `summarize_table`, `create_all_summaries`) | Medium |
| 7 | `routers/pdfs.py` | Add `create_multimodal_retriever()` function | Medium |
| 8 | `routers/pdfs.py` | Replace `create_pdf_rag_agent()` with `create_multimodal_rag_agent()` | Easy |
| 9 | `routers/pdfs.py` | Update the `/ask` endpoint | Easy |
| 10 | Frontend (optional) | Update placeholder text | Trivial |

### Files That DON'T Change

| File | Why No Changes |
|------|----------------|
| `main.py` | Just imports and starts the server |
| `database.py` | Database connection stays the same |
| `models.py` | PDF table structure stays the same |
| `schemas.py` | Request/response format stays the same |
| `crud.py` | CRUD operations stay the same |
| Frontend `pdf.js` | Same Q&A interface |
| Frontend `pdf-list.js` | Same PDF list |
| Frontend styles | Same styling |

### Files That DO Change

| File | What Changed |
|------|-------------|
| `pyproject.toml` | Added `unstructured-client`, `langchain-unstructured`, `langchain-chroma`, `pillow`, `pymupdf` |
| `.env` | Added `UNSTRUCTURED_API_KEY` |
| `config.py` | Added `UNSTRUCTURED_API_KEY: str` setting |
| `routers/pdfs.py` | Major rewrite of the multimodal RAG logic (see sections above) |

### The Data Flow: Before vs After

**BEFORE (Regular RAG)**:
```
PDF --> PyPDFLoader --> Text chunks --> Embeddings --> InMemoryVectorStore --> Agent --> Answer
```

**AFTER (Multimodal RAG)**:
```
         +-> Unstructured API -+-- Text   --> Summarize (GPT-4o-mini) --+
         |                     |                                        |
PDF -----+                     +-- Tables --> Summarize (GPT-4o-mini) --+--> MultiVectorRetriever --> Agent --> Answer
         |                                                              |
         +-> pymupdf (pages) ---- Images  --> Describe  (GPT-4o-mini) --+
```

### What the User Experiences

From the user's perspective, the app looks exactly the same! The only difference is in what it can answer:

| Question | Before (Regular RAG) | After (Multimodal RAG) |
|----------|---------------------|------------------------|
| "What is this document about?" | Can answer | Can answer |
| "What are the total expenses?" | Can answer (if in text) | Can answer (even if in a table) |
| "What does the chart show?" | **Cannot answer** | Can answer! |
| "Describe the diagram on page 5" | **Cannot answer** | Can answer! |
| "What product is shown in the images?" | **Cannot answer** | Can answer! |

---
## Key Takeaways

1. **The frontend doesn't need to change** because it just sends questions and receives text answers. The multimodal magic happens entirely in the backend.

2. **The main backend change uses a two-tool approach for PDF extraction**: the Unstructured API (via `unstructured-client`) extracts text and tables, while `pymupdf` renders each page as an image. This combination ensures nothing is missed — even vector-based charts that the Unstructured API cannot detect as images.

3. **Page images are handled by describing them in text** using GPT-4o's vision capabilities. pymupdf renders each page as a PNG, base64-encodes it, and sends it to GPT-4o for a detailed description.

4. **The Multi-Vector Retriever uses two stores**: summaries for searching (fast) and originals for answering (accurate).

5. **Most files don't change at all**: `main.py`, `database.py`, `models.py`, `schemas.py`, `crud.py` all stay exactly the same. The changes are in `config.py` (new API key setting), `routers/pdfs.py` (multimodal RAG logic), and `pyproject.toml` (new dependencies including `pymupdf`).

6. **The API contract stays the same**: The `/pdfs/{id}/ask` endpoint still accepts `{"question": "..."}` and returns `{"answer": "..."}`. This is why the frontend doesn't need changes.

7. **In production**, you would want to process PDFs at upload time and cache the retriever, rather than re-processing on every question. This is an optimization left as an exercise.

---
## Congratulations!

You now understand how to upgrade a full-stack RAG application to handle multimodal content! The key insight is that **good software architecture makes upgrades easier**: because our app had a clean separation between frontend and backend, and a well-defined API contract, we only needed to change the internal implementation of one file (`routers/pdfs.py`) to add a powerful new capability.

This is a fundamental principle of software engineering: **program to interfaces, not implementations**. The frontend programs to the `/ask` API interface, so it doesn't care if the implementation behind it changes from regular RAG to multimodal RAG.